In [ ]:
import json
from collections import defaultdict

# Load JSON
with open("train_val_videodatainfo.json", 'r') as f:
    data = json.load(f)

# Map video_id to all its captions
video_captions = defaultdict(list)
for entry in data['sentences']:
    video_captions[entry['video_id']].append(entry['caption'])

# Optional: Filter videos for training
train_videos = {v['video_id'] for v in data['videos'] if v['split'] == 'train'}
train_captions = {vid: video_captions[vid] for vid in train_videos}


In [ ]:
import cv2
import os

def extract_frames(video_path, num_frames=10):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // num_frames, 1)

    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, frame = cap.read()
        if success:
            frame = cv2.resize(frame, (224, 224))  # Resize for CNN
            frames.append(frame)
        if len(frames) == num_frames:
            break
    cap.release()
    return frames


In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms

resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove classification head
resnet.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def get_features(frames):
    inputs = torch.stack([transform(frame) for frame in frames])
    with torch.no_grad():
        features = resnet(inputs)
    return features.squeeze()  # Shape: (N_frames, 2048)


In [ ]:
from collections import Counter

# Build vocab
all_captions = [cap for caps in train_captions.values() for cap in caps]
counter = Counter(word for cap in all_captions for word in cap.lower().split())
vocab = ['<PAD>', '<SOS>', '<EOS>', '<UNK>'] + [word for word, count in counter.items() if count >= 5]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}


In [ ]:
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        modules = list(resnet.children())[:-1]  # Remove last fc
        self.resnet = nn.Sequential(*modules)
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, x):  # x: (B, N, 3, 224, 224)
        batch_size, n_frames, C, H, W = x.shape
        x = x.view(batch_size * n_frames, C, H, W)
        features = self.resnet(x)  # (B*N, 2048, 1, 1)
        features = features.view(batch_size, n_frames, -1)  # (B, N, 2048)
        return features

In [ ]:
class Decoder(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size + 2048, hidden_size, num_layers,
                            batch_first=True, bidirectional=True)
        self.linear = nn.Linear(hidden_size * 2, vocab_size)  # note: hidden_size × 2

    def forward(self, features, captions):
        embeddings = self.embedding(captions)  # (B, T, embed)
        video_feat = features.mean(1).unsqueeze(1).expand(-1, captions.size(1), -1)  # (B, T, 2048)
        lstm_input = torch.cat((embeddings, video_feat), dim=2)  # (B, T, embed+2048)
        hiddens, _ = self.lstm(lstm_input)
        outputs = self.linear(hiddens)  # (B, T, vocab_size)
        return outputs


In [ ]:
class VideoCaptioningModel(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size):
        super(VideoCaptioningModel, self).__init__()
        self.encoder = EncoderCNN()
        self.decoder = Decoder(embed_size, hidden_size, vocab_size)

    def forward(self, frames, captions):
        features = self.encoder(frames)
        outputs = self.decoder(features, captions)
        return outputs


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import cv2
import random

class MSRVTTDataset(Dataset):
    def __init__(self, video_folder, video_captions, word2idx, transform, max_len=20):
        self.video_folder = video_folder
        self.video_captions = video_captions
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len
        self.video_ids = list(video_captions.keys())

    def __len__(self):
        return len(self.video_ids)

    def extract_frames(self, video_path, num_frames=4):  # Reduce for speed
        cap = cv2.VideoCapture(video_path)
        frames = []
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total // num_frames)
        for i in range(0, total, step):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                if self.transform:
                    frame = self.transform(frame)
                frames.append(frame)
                if len(frames) == num_frames:
                    break
        cap.release()
        return torch.stack(frames)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]
        video_path = os.path.join(self.video_folder, f"{video_id}.mp4")
        frames = self.extract_frames(video_path)

        # ✅ Random caption from list
        caption = random.choice(self.video_captions[video_id])
        tokens = ["<SOS>"] + caption.lower().strip().split() + ["<EOS>"]
        caption_indices = [self.word2idx.get(word, self.word2idx["<UNK>"]) for word in tokens]
        caption_indices += [self.word2idx["<PAD>"]] * (self.max_len - len(caption_indices))
        caption_indices = caption_indices[:self.max_len]

        return frames, torch.tensor(caption_indices)


In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishnutheepb/msrvtt")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/msrvtt


In [ ]:
import os

for item in os.listdir("/kaggle/input/msrvtt"):
    print(item)


TrainValVideo


In [ ]:
import os

for item in os.listdir("/kaggle/input/msrvtt/TrainValVideo"):
    print(item)


video2438.mp4
video797.mp4
video3708.mp4
video5598.mp4
video1806.mp4
video4504.mp4
video1521.mp4
video4853.mp4
video6165.mp4
video1906.mp4
video1750.mp4
video6249.mp4
video1239.mp4
video2188.mp4
video1442.mp4
video502.mp4
video3744.mp4
video1107.mp4
video3882.mp4
video6604.mp4
video1671.mp4
video330.mp4
video2540.mp4
video2284.mp4
video4913.mp4
video3240.mp4
video1283.mp4
video5107.mp4
video2842.mp4
video6295.mp4
video2532.mp4
video1064.mp4
video1248.mp4
video892.mp4
video6530.mp4
video3360.mp4
video5640.mp4
video4491.mp4
video6318.mp4
video4433.mp4
video1297.mp4
video2583.mp4
video4101.mp4
video5730.mp4
video3171.mp4
video1533.mp4
video5609.mp4
video1941.mp4
video3465.mp4
video3291.mp4
video2288.mp4
video379.mp4
video909.mp4
video5778.mp4
video4929.mp4
video6967.mp4
video1033.mp4
video2496.mp4
video6811.mp4
video5821.mp4
video6357.mp4
video6622.mp4
video4756.mp4
video1060.mp4
video4030.mp4
video2890.mp4
video19.mp4
video858.mp4
video3414.mp4
video1327.mp4
video3876.mp4
video3952.mp4
v

In [ ]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VideoCaptioningModel(embed_size=256, hidden_size=512, vocab_size=len(word2idx)).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])  # ignore padding tokens
optimizer = optim.Adam(model.parameters(), lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 116MB/s]


In [ ]:
# Slice the first 200 video-caption pairs from the JSON
sliced_captions = dict(list(train_captions.items())[:200])


In [ ]:
dataset = MSRVTTDataset(
    video_folder="/kaggle/input/msrvtt/TrainValVideo",
    video_captions=sliced_captions,  # ✅ use sliced subset
    word2idx=word2idx,
    transform=transform,
    max_len=20
)


In [ ]:
import random

video_ids = list(train_captions.keys())
random.shuffle(video_ids)
subset_ids = video_ids[:200]

sliced_captions = {vid: train_captions[vid] for vid in subset_ids}


In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [ ]:
import random

video_ids = list(train_captions.keys())
random.shuffle(video_ids)
subset_ids = video_ids[:200]

sliced_captions = {vid: train_captions[vid] for vid in subset_ids}


In [ ]:
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model, Loss, Optimizer
model = VideoCaptioningModel(embed_size=256, hidden_size=512, vocab_size=len(word2idx)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Number of epochs
num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for frames, captions in loop:
        frames = frames.to(device)            # (B, N, 3, 224, 224)
        captions = captions.to(device)        # (B, T)

        optimizer.zero_grad()

        outputs = model(frames, captions[:, :-1])  # Predict next token
        loss = criterion(
            outputs.reshape(-1, outputs.shape[-1]),  # (B*T, vocab)
            captions[:, 1:].reshape(-1)              # (B*T)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader)
    print(f"✅ Epoch {epoch+1} completed. Avg Loss: {avg_loss:.4f}")


Epoch 1/10: 100%|██████████| 100/100 [06:04<00:00,  3.65s/it, loss=4.1]


✅ Epoch 1 completed. Avg Loss: 6.7964


Epoch 2/10: 100%|██████████| 100/100 [06:08<00:00,  3.68s/it, loss=5.82]


✅ Epoch 2 completed. Avg Loss: 5.2073


Epoch 3/10: 100%|██████████| 100/100 [06:11<00:00,  3.72s/it, loss=5.6]


...

✅ Epoch 8 completed. Avg Loss: 3.6766


Epoch 9/10: 100%|██████████| 100/100 [06:07<00:00,  3.67s/it, loss=2.41]


✅ Epoch 9 completed. Avg Loss: 3.5041


Epoch 10/10: 100%|██████████| 100/100 [06:06<00:00,  3.66s/it, loss=3.11]

✅ Epoch 10 completed. Avg Loss: 3.4371


In [ ]:
def generate_caption(model, frames, word2idx, idx2word, max_len=20, device='cpu', temperature=1.0):
    model.eval()
    with torch.no_grad():
        frames = frames.to(device).unsqueeze(0)  # (1, N, 3, 224, 224)
        features = model.encoder(frames)         # (1, N, 2048)
        inputs = torch.tensor([word2idx['<SOS>']], device=device).unsqueeze(0)  # (1, 1)

        caption = []
        states = None

        for _ in range(max_len):
            embedded = model.decoder.embedding(inputs)  # (1, 1, embed_size)
            video_feat = features.mean(1).unsqueeze(1)  # (1, 1, 2048)
            lstm_input = torch.cat((embedded, video_feat), dim=2)
            hiddens, states = model.decoder.lstm(lstm_input, states)
            output = model.decoder.linear(hiddens.squeeze(1))  # (1, vocab_size)

            # ✅ Apply temperature and sample
            probs = torch.softmax(output / temperature, dim=1)
            predicted = torch.multinomial(probs, 1).item()

            word = idx2word.get(predicted, "<UNK>")
            if word == "<EOS>":
                break
            caption.append(word)
            inputs = torch.tensor([[predicted]], device=device)

        return ' '.join(caption)


In [ ]:
torch.save(model.state_dict(), "video_caption_model.pth")


In [ ]:
model.load_state_dict(torch.load("video_caption_model.pth"))
model.eval()


VideoCaptioningModel(
  (encoder): EncoderCNN(
    (resnet): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=T

In [ ]:
frames, _ = dataset[0]  # get sample video
caption = generate_caption(model, frames, word2idx, idx2word, device=device, temperature=0.8)
print("📝 Generated Caption:", caption)


📝 Generated Caption: passing restuarant military receipee
